# Carver A/B arms — target geometry and bucket-frequency policy

Four carving arms on the **same library, same machine, same
train/dev split** as `frequency_model_2026.ipynb`, whose setup cells below are reused
verbatim. Each arm changes exactly one thing, so the difference is attributable.

| Arm | Carver | Isolates |
|---|---|---|
| **A1** | `OneVsRestCarver` | the 2025 geometry — one binary carving per target class, so a raw feature becomes several carved columns |
| **A2** | `MulticlassCarver` | one carving per feature against the full crosstab, target treated as **unordered** |
| **A3** | `OrdinalCarver(target_scale="level")` | the shipped 2026 arm — one carving per feature, target treated as **ordered** (Kendall's tau-c) |
| **A4** | A3 with `min_freq_alpha=1.0` | **Wilson-score CI off.** At alpha=1 the z-score is 0, so the Wilson upper bound collapses to the raw proportion — i.e. the pre-7.x hard `count/nobs < min_freq` cutoff |

**The common yardstick is post-hoc tau-c on the dev set.** Each carver optimises its own
association measure, so their internal scores are not comparable across arms. Kendall's
tau-c, on the other hand, is defined for *any* ordered binning against the ordinal target
`0 < 1 < 2+`, whoever produced the binning. Every arm is scored the same way, with the
library's own `AutoCarver.stats.rank_associations`.

Two aggregates matter and they answer different questions:

* **best tau-c per raw feature** — how much ordinal signal that feature's carving retained,
  regardless of how many columns it took;
* **columns per raw feature** — what that retention cost in feature-matrix width.

A1 can win the first while losing badly on the second. That trade is the point.

Structural pass only: no selector, no Optuna, no dev metric. Wall-clocks are recorded but
are *not* the article's speed headline (that comes from the 7.0.5 baselines) — here they
are context for the amount of work each geometry does.

## Loading data & target  *(reused from `frequency_model_2026.ipynb`)*

In [1]:
import pandas as pd

data_path = "../data/"

# loading x_train
data = pd.read_csv(data_path + "train_input_Z61KlZo.csv", low_memory=False)
data.set_index("ID", inplace=True)
print("x_train", data.shape)

# loading target
target = pd.read_csv(data_path + "train_output_DzPxaPY.csv", low_memory=False)
target.set_index("ID", inplace=True)
print("y_train", target.shape)

# joining x_train and y_train
data = data.join(target.drop("ANNEE_ASSURANCE", axis=1))
print("data", data.shape)

x_train (383610, 373)
y_train (383610, 4)


data (383610, 376)


In [2]:
target_col = "TARGET"
data[target_col] = (data["FREQ"] * data["ANNEE_ASSURANCE"]).astype(int)
data[target_col].value_counts(normalize=False).sort_index()

TARGET
0    381061
1      2458
2        87
3         2
4         1
5         1
Name: count, dtype: int64

## Stratified sampling & inverse-frequency weights  *(reused)*

In [3]:
import numpy as np

from collections import Counter
from sklearn.model_selection import train_test_split
from utils.data_toolkit import collapse_count

# Compute class frequencies
class_counts = Counter(collapse_count(data[target_col]))
total_samples = len(data[target_col])

# Compute inverse frequency class weights
class_weights = {
    cls: total_samples / (len(class_counts) * count)
    for cls, count in class_counts.items()
}
print("Class weights:", class_weights)

# Assign sample weights based on target values
weights = np.array([class_weights[label] for label in collapse_count(data[target_col])])

# Train-test split
x_train, x_dev, y_train, y_dev, w_train, w_dev = train_test_split(
    data,
    data[target_col],
    weights,
    test_size=0.2,
    random_state=42,
    stratify=collapse_count(data[target_col]),
)
print("y_train mean", y_train.mean(), " observations:", x_train.shape[0])
print("y_dev mean", y_dev.mean(), " observations:", x_dev.shape[0])

Class weights: {0: 0.3355630725789309, 1: 52.0219690805533, 2: 1405.1648351648353}


y_train mean 0.006895023591668622  observations: 306888
y_dev mean 0.006921091733792133  observations: 76722


## Feature engineering  *(reused)*

In [4]:
from utils.data_toolkit import Processor

proc = Processor()
x_train = proc.fit_transform(x_train)
x_dev = proc.transform(x_dev)
data = proc.transform(data)

## Column typing  *(reused)*

In [5]:
from AutoCarver import Features

# sorting columns per type
features = Features.from_dataframe(x_train)

# getting ordinal columns
ordinals = [
    "NB_CASERNES",
    "BDTOPO_BAT_MAX_HAUTEUR",
    "HAUTEUR_MAX",
    "HAUTEUR",
    "BDTOPO_BAT_MAX_HAUTEUR_MAX",
    "MEN_SURF",
    "IND_SNV",
    "IND_INC",
    "IND_Y9",
    "IND_0_Y1",
    "IND",
    "LOG_SOC",
    "LOG_INC",
    "LOG_APA3",
    "LOG_AVA1",
    "MEN_MAIS",
    "MEN_COLL",
    "MEN_FMP",
    "MEN_PROP",
    "MEN_PAUV",
    "MEN",
    "COEFASS",
]
ordinals += [
    "DISTANCE_111",
    "DISTANCE_112",
    "DISTANCE_121",
    "DISTANCE_122",
    "DISTANCE_123",
    "DISTANCE_124",
    "DISTANCE_131",
    "DISTANCE_132",
    "DISTANCE_133",
    "DISTANCE_141",
    "DISTANCE_142",
    "DISTANCE_211",
    "DISTANCE_212",
    "DISTANCE_213",
    "DISTANCE_221",
    "DISTANCE_222",
    "DISTANCE_223",
    "DISTANCE_231",
    "DISTANCE_242",
    "DISTANCE_243",
    "DISTANCE_244",
    "DISTANCE_311",
    "DISTANCE_312",
    "DISTANCE_313",
    "DISTANCE_321",
    "DISTANCE_322",
    "DISTANCE_323",
    "DISTANCE_324",
    "DISTANCE_331",
    "DISTANCE_332",
    "DISTANCE_333",
    "DISTANCE_334",
    "DISTANCE_335",
    "DISTANCE_411",
    "DISTANCE_412",
    "DISTANCE_421",
    "DISTANCE_422",
    "DISTANCE_423",
    "DISTANCE_511",
    "DISTANCE_512",
    "DISTANCE_521",
    "DISTANCE_522",
    "DISTANCE_523",
    "PROPORTION_11",
    "PROPORTION_12",
    "PROPORTION_13",
    "PROPORTION_14",
    "PROPORTION_21",
    "PROPORTION_22",
    "PROPORTION_23",
    "PROPORTION_24",
    "PROPORTION_31",
    "PROPORTION_32",
    "PROPORTION_33",
    "PROPORTION_41",
    "PROPORTION_42",
    "PROPORTION_51",
    "PROPORTION_52",
    "MEN_1IND",
    "MEN_5IND",
    "LOG_A1_A2",
    "LOG_A2_A3",
    "IND_Y1_Y2",
    "IND_Y2_Y3",
    "IND_Y3_Y4",
    "IND_Y4_Y5",
    "IND_Y5_Y6",
    "IND_Y6_Y7",
    "IND_Y7_Y8",
    "IND_Y8_Y9",
    "DISTANCE_1",
    "DISTANCE_2",
    "ALTITUDE_1",
    "ALTITUDE_2",
    "ALTITUDE_3",
    "ALTITUDE_4",
    "ALTITUDE_5",
    "NBJTX25_MM_A",
    "NBJTX25_MMAX_A",
    "NBJTX25_MSOM_A",
    "NBJTX0_MM_A",
    "NBJTX0_MMAX_A",
    "NBJTX0_MSOM_A",
    "NBJTXI27_MM_A",
    "NBJTXI27_MMAX_A",
    "NBJTXI27_MSOM_A",
    "NBJTXS32_MM_A",
    "NBJTXS32_MMAX_A",
    "NBJTXS32_MSOM_A",
    "NBJTXI20_MM_A",
    "NBJTXI20_MMAX_A",
    "NBJTXI20_MSOM_A",
    "NBJTX30_MM_A",
    "NBJTX30_MMAX_A",
    "NBJTX30_MSOM_A",
    "NBJTX35_MM_A",
    "NBJTX35_MMAX_A",
    "NBJTX35_MSOM_A",
    "NBJTN10_MM_A",
    "NBJTN10_MMAX_A",
    "NBJTN10_MSOM_A",
    "NBJTNI10_MM_A",
    "NBJTNI10_MMAX_A",
    "NBJTNI10_MSOM_A",
    "NBJTN5_MM_A",
    "NBJTN5_MMAX_A",
    "NBJTN5_MSOM_A",
    "NBJTNS25_MM_A",
    "NBJTNS25_MMAX_A",
    "NBJTNS25_MSOM_A",
    "NBJTNI15_MM_A",
    "NBJTNI15_MMAX_A",
    "NBJTNI15_MSOM_A",
    "NBJTNI20_MM_A",
    "NBJTNI20_MMAX_A",
    "NBJTNI20_MSOM_A",
    "NBJTNS20_MM_A",
    "NBJTNS20_MMAX_A",
    "NBJTNS20_MSOM_A",
    "NBJTMS24_MM_A",
    "NBJTMS24_MMAX_A",
    "NBJTMS24_MSOM_A",
    "TAMPLIAB_VOR_MM_A",
    "TAMPLIAB_VOR_MMAX_A",
    "TAMPLIM_VOR_MM_A",
    "TAMPLIM_VOR_MMAX_A",
    "TM_VOR_MM_A",
    "TM_VOR_MMAX_A",
    "TMM_VOR_MM_A",
    "TMM_VOR_MMAX_A",
    "TMMAX_VOR_MM_A",
    "TMMAX_VOR_MMAX_A",
    "TMMIN_VOR_MM_A",
    "TMMIN_VOR_MMAX_A",
    "TN_VOR_MM_A",
    "TN_VOR_MMAX_A",
    "TNAB_VOR_MM_A",
    "TNAB_VOR_MMAX_A",
    "TNMAX_VOR_MM_A",
    "TNMAX_VOR_MMAX_A",
    "TX_VOR_MM_A",
    "TX_VOR_MMAX_A",
    "TXAB_VOR_MM_A",
    "TXAB_VOR_MMAX_A",
    "TXMIN_VOR_MM_A",
    "TXMIN_VOR_MMAX_A",
    "NBJFF10_MM_A",
    "NBJFF10_MMAX_A",
    "NBJFF10_MSOM_A",
    "NBJFF16_MM_A",
    "NBJFF16_MMAX_A",
    "NBJFF16_MSOM_A",
    "NBJFF28_MM_A",
    "NBJFF28_MMAX_A",
    "NBJFF28_MSOM_A",
    "NBJFXI3S10_MM_A",
    "NBJFXI3S10_MMAX_A",
    "NBJFXI3S10_MSOM_A",
    "NBJFXI3S16_MM_A",
    "NBJFXI3S16_MMAX_A",
    "NBJFXI3S16_MSOM_A",
    "NBJFXI3S28_MM_A",
    "NBJFXI3S28_MMAX_A",
    "NBJFXI3S28_MSOM_A",
    "NBJFXY8_MM_A",
    "NBJFXY8_MMAX_A",
    "NBJFXY8_MSOM_A",
    "NBJFXY10_MM_A",
    "NBJFXY10_MMAX_A",
    "NBJFXY10_MSOM_A",
    "NBJFXY15_MM_A",
    "NBJFXY15_MMAX_A",
    "NBJFXY15_MSOM_A",
    "FFM_VOR_MM_A",
    "FFM_VOR_MMAX_A",
    "FXI3SAB_VOR_MM_A",
    "FXI3SAB_VOR_MMAX_A",
    "FXIAB_VOR_MM_A",
    "FXIAB_VOR_MMAX_A",
    "FXYAB_VOR_MM_A",
    "FXYAB_VOR_MMAX_A",
    "FFM_VOR_COM_MM_A_Y",
    "FFM_VOR_COM_MMAX_A_Y",
    "FXI3SAB_VOR_COM_MM_A_Y",
    "FXI3SAB_VOR_COM_MMAX_A_Y",
    "NBJRR50_MM_A",
    "NBJRR50_MMAX_A",
    "NBJRR50_MSOM_A",
    "NBJRR1_MM_A",
    "NBJRR1_MMAX_A",
    "NBJRR1_MSOM_A",
    "NBJRR5_MM_A",
    "NBJRR5_MMAX_A",
    "NBJRR5_MSOM_A",
    "NBJRR10_MM_A",
    "NBJRR10_MMAX_A",
    "NBJRR10_MSOM_A",
    "NBJRR30_MM_A",
    "NBJRR30_MMAX_A",
    "NBJRR30_MSOM_A",
    "NBJRR100_MM_A",
    "NBJRR100_MMAX_A",
    "NBJRR100_MSOM_A",
    "RR_VOR_MM_A",
    "RR_VOR_MMAX_A",
    "RRAB_VOR_MM_A",
    "RRAB_VOR_MMAX_A",
]
ordinals += ["TAILLE1", "TAILLE2"]
ordinal_columns = {
    col: list(data[col].value_counts().sort_index().index)
    for col in ordinals
    if col in data.columns
}
ordinal_columns["PROPORTION_32"] += ["10. > 90"]
ordinal_columns.update(
    {
        "CARACT4": [
            "absence de surface",
            "Surface de moins d",
            "Surface entre 501",
            "Surface entre 1001",
            "Surface entre 1501",
            "Surface de plus de",
        ],
        "SURFACE4": [
            "0",
            "500",
            "1000",
            "1500",
            "2000",
            "2500",
            "3000",
            "3500",
            "4000",
            "4500",
            "5000",
            "5500",
            "6000",
            "6500",
            "7000",
            "7000+",
        ],
        "SURFACE6": [
            "0",
            "500",
            "1000",
            "1500",
            "2000",
            "2500",
            "3000",
            "3500",
            "4000",
            "4500",
            "5000",
            "5500",
            "6000",
            "6500",
            "7000",
            "7000+",
        ],
        "total_surface_2023": [
            "Aucun feu",
            "<10ha",
            "10-20ha",
            "20-50ha",
            "50-100ha",
            "100-200ha",
            ">200ha",
        ],
        "total_surface_5y": [
            "Aucun feu",
            "<10ha",
            "10-20ha",
            "20-50ha",
            "50-100ha",
            "100-200ha",
            ">200ha",
        ],
        "surface_over_forest": [
            "Absence de feu",
            "<0.05",
            "0.05-0.1",
            "0.1-0.2",
            "0.5-2",
        ],
        "fire_extinction_rates": ["Aucun feu", "<50%", "50-70%", "70-85%", ">85%"],
    }
)

# columns that are to be removed (target + no values)
to_remove = target.columns.tolist() + [target_col]
to_remove += [c for c in data.columns if "MMSOM" in c]
to_remove += ["DEROG13", "DEROG14", "DEROG16"]

# removing columns
categorical_columns = [
    col.name
    for col in features.categoricals
    if col not in to_remove and col not in ordinal_columns
]
categorical_columns += ["TYPERS"]
numerical_columns = [
    col.name
    for col in features.numericals
    if col not in to_remove
    and col not in ordinal_columns
    and col not in categorical_columns
]
print(
    len(categorical_columns),
    len(numerical_columns),
    len(ordinal_columns),
    len(categorical_columns) + len(numerical_columns) + len(ordinal_columns),
)

197 120 238 555


## The arms

`min_freq` is left at the library default (0.02) and `max_n_mod` at 5 for every arm, matching
`frequency_model_2026.ipynb`. `dropna=False`, so `NaN` is carved as its own bucket.

`OneVsRestCarver` forces `copy=True` internally (it cannot assign in place); the others run
`copy=False` against a fresh copy of the training frame, so no arm sees another's output.

In [6]:
import gc
from time import perf_counter

import pandas as pd

from AutoCarver import Features, MulticlassCarver, OneVsRestCarver, OrdinalCarver
from AutoCarver.discretizers import ProcessingConfig
from AutoCarver.stats import rank_associations

N_JOBS = 6

# the 0/1/2+ claim-count target, as in frequency_model_2026.ipynb
y_ord_train = collapse_count(y_train)
y_ord_dev = collapse_count(y_dev)


def score_arm(carver, x_dev_carved, y_ord):
    """post-hoc tau-c on dev, per carved column, on the common ordinal target"""
    rows = []
    for feature in carver.features:
        column = feature.version
        if column not in x_dev_carved.columns:
            continue
        crosstab = pd.crosstab(x_dev_carved[column], y_ord)
        # rows = buckets ascending (carvers ordinal-encode), columns = target levels ascending
        association = rank_associations(crosstab.values)
        rows.append(
            {
                "feature": feature.name,
                "column": column,
                "n_buckets": crosstab.shape[0],
                "tau_c": association.get("tau_c"),
            }
        )
    return pd.DataFrame(rows)


def run_arm(label, build_carver, min_freq_alpha=0.05):
    qualitatives = Features(categoricals=categorical_columns, ordinals=ordinal_columns)
    n_input = len(qualitatives)
    config = ProcessingConfig(
        dropna=False,
        copy=False,
        verbose=False,
        n_jobs=N_JOBS,
        min_freq_alpha=min_freq_alpha,
    )
    carver = build_carver(qualitatives, config)

    # fresh frames so no arm inherits another's carving. OneVsRestCarver forces
    # copy=True internally, so copying again here would just double peak memory.
    xt = x_train if isinstance(carver, OneVsRestCarver) else x_train.copy()
    xd = x_dev.copy()

    start = perf_counter()
    xt = carver.fit_transform(xt, y_ord_train, X_dev=xd, y_dev=y_ord_dev)
    elapsed = perf_counter() - start
    xd = carver.transform(xd)

    scores = score_arm(carver, xd, y_ord_dev)
    scores.insert(0, "arm", label)

    kept = scores["feature"].nunique()
    summary = {
        "arm": label,
        "carver": type(carver).__name__,
        "min_freq_alpha": min_freq_alpha,
        "carve_seconds": round(elapsed, 1),
        "features_in": n_input,
        "features_kept": kept,
        "features_dropped": n_input - kept,
        "columns_out": len(scores),
        "columns_per_feature": round(len(scores) / kept, 3) if kept else None,
        "buckets_total": int(scores["n_buckets"].sum()),
        "buckets_mean": round(scores["n_buckets"].mean(), 3),
        "buckets_median": float(scores["n_buckets"].median()),
        "tau_c_mean": round(scores["tau_c"].mean(), 6),
        "tau_c_median": round(scores["tau_c"].median(), 6),
        "tau_c_best_per_feature_mean": round(
            scores.groupby("feature")["tau_c"].max().mean(), 6
        ),
        "tau_c_top100_sum": round(
            scores["tau_c"].abs().sort_values(ascending=False).head(100).sum(), 6
        ),
    }
    print(
        f"[{label}] {summary['carver']} alpha={min_freq_alpha} "
        f"{summary['carve_seconds']}s on {N_JOBS} workers | "
        f"{summary['columns_out']} columns from {kept} features "
        f"({summary['columns_per_feature']}/feature) | "
        f"dropped {summary['features_dropped']} | "
        f"buckets {summary['buckets_total']} | "
        f"tau_c mean {summary['tau_c_mean']}"
    )

    del xt, xd, carver
    gc.collect()
    return summary, scores

In [7]:
ARMS = [
    ("A1 one-vs-rest", lambda f, c: OneVsRestCarver(features=f, config=c), 0.05),
    ("A2 multiclass", lambda f, c: MulticlassCarver(features=f, config=c), 0.05),
    (
        "A3 ordinal",
        lambda f, c: OrdinalCarver(features=f, target_scale="level", config=c),
        0.05,
    ),
    (
        "A4 ordinal, Wilson off",
        lambda f, c: OrdinalCarver(features=f, target_scale="level", config=c),
        1.0,
    ),
]

summaries, details = [], []
for label, build, alpha in ARMS:
    summary, scores = run_arm(label, build, min_freq_alpha=alpha)
    summaries.append(summary)
    details.append(scores)

summary_table = pd.DataFrame(summaries).set_index("arm")
detail_table = pd.concat(details, ignore_index=True)

[BinaryCarver] Carving:   0%|          | 0/435 [00:00<?, ?feature/s]

[BinaryCarver] dropped 44/435 feature(s) (no robust train/dev combination): KAPITAL34, LOG_VETUSTE_REGION, MEN_num, ADOSS, IND_num, IND_Y1_Y2_num, IND_0_Y1_num, IND_Y6_Y7_num, IND_Y7_Y8_num, IND_INC_num, IND_0_Y1_IND, IND_Y1_Y2_IND, IND_TOT, IND_Y2_Y3_IND, IND_0_Y1_IND_TOT, IND_Y1_Y2_IND_TOT, IND_Y6_Y7_IND, IND_Y7_Y8_IND, IND_Y2_Y3_IND_TOT, IND_INC_IND, IND_Y4_Y5_IND_TOT, IND_INC_IND_TOT, IND_INC, IND_0_Y1, IND, LOG_INC, LOG_SOC, MEN, PROPORTION_12, PROPORTION_14, PROPORTION_13, PROPORTION_11, PROPORTION_41, PROPORTION_33, PROPORTION_42, PROPORTION_51, PROPORTION_52, IND_Y1_Y2, IND_Y2_Y3, IND_Y4_Y5, LOG_A1_A2, IND_Y6_Y7, IND_Y7_Y8, LOG_VETUSTE


C:\Users\defra\Desktop\git\PROJECTS\caa-challenge\.venv\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:548: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mapped = mapped.fillna(col)
C:\Users\defra\Desktop\git\PROJECTS\caa-challenge\.venv\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:548: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mapped = mapped.fillna(col)
C:\Users\defra\Desktop\git\PROJECTS\caa-challenge\.venv\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:548: FutureWarning: Downcastin

C:\Users\defra\Desktop\git\PROJECTS\caa-challenge\.venv\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:548: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mapped = mapped.fillna(col)
C:\Users\defra\Desktop\git\PROJECTS\caa-challenge\.venv\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:548: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mapped = mapped.fillna(col)
C:\Users\defra\Desktop\git\PROJECTS\caa-challenge\.venv\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:548: FutureWarning: Downcastin

C:\Users\defra\Desktop\git\PROJECTS\caa-challenge\.venv\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:548: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mapped = mapped.fillna(col)
C:\Users\defra\Desktop\git\PROJECTS\caa-challenge\.venv\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:548: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mapped = mapped.fillna(col)
C:\Users\defra\Desktop\git\PROJECTS\caa-challenge\.venv\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:548: FutureWarning: Downcastin

...

[OrdinalCarver] dropped 38/435 feature(s) (no robust train/dev combination): LOG_VETUSTE_REGION, MEN_num, IND_num, IND_0_Y1_num, IND_Y1_Y2_num, IND_Y4_Y5_num, IND_Y6_Y7_num, IND_Y7_Y8_num, IND_INC_num, IND_0_Y1_IND, IND_Y1_Y2_IND, IND_Y2_Y3_IND, IND_Y6_Y7_IND, IND_Y7_Y8_IND, IND_INC_IND, IND_INC, IND_0_Y1, IND, LOG_INC, MEN, PROPORTION_13, PROPORTION_12, PROPORTION_14, PROPORTION_11, PROPORTION_33, PROPORTION_41, PROPORTION_42, PROPORTION_51, PROPORTION_52, IND_Y1_Y2, IND_Y2_Y3, LOG_A1_A2, IND_Y4_Y5, IND_Y6_Y7, IND_Y7_Y8, LOG_APA3_num_LOG_TOT, LOG_AVA1_num_LOG_TOT, LOG_VETUSTE


[A3 ordinal] OrdinalCarver alpha=0.05 126.8s on 6 workers | 397 columns from 397 features (1.0/feature) | dropped 38 | buckets 834 | tau_c mean 0.00059


[OrdinalCarver] Carving:   0%|          | 0/435 [00:00<?, ?feature/s]

[OrdinalCarver] dropped 38/435 feature(s) (no robust train/dev combination): LOG_VETUSTE_REGION, MEN_num, IND_num, IND_0_Y1_num, IND_Y1_Y2_num, IND_Y4_Y5_num, IND_Y6_Y7_num, IND_Y7_Y8_num, IND_INC_num, IND_0_Y1_IND, IND_Y1_Y2_IND, IND_Y2_Y3_IND, IND_Y6_Y7_IND, IND_Y7_Y8_IND, IND_INC_IND, IND_INC, IND_0_Y1, IND, LOG_INC, MEN, PROPORTION_13, PROPORTION_12, PROPORTION_14, PROPORTION_11, PROPORTION_33, PROPORTION_41, PROPORTION_42, PROPORTION_51, PROPORTION_52, IND_Y1_Y2, IND_Y2_Y3, IND_Y4_Y5, LOG_A1_A2, IND_Y6_Y7, IND_Y7_Y8, LOG_APA3_num_LOG_TOT, LOG_AVA1_num_LOG_TOT, LOG_VETUSTE


[A4 ordinal, Wilson off] OrdinalCarver alpha=1.0 118.7s on 6 workers | 397 columns from 397 features (1.0/feature) | dropped 38 | buckets 833 | tau_c mean 0.00059


## Results

In [8]:
from pathlib import Path

out_dir = Path("../data/ab_arms")
out_dir.mkdir(parents=True, exist_ok=True)
summary_table.to_csv(out_dir / "summary.csv")
detail_table.to_csv(out_dir / "per_column_tau_c.csv", index=False)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
summary_table

,carver,min_freq_alpha,carve_seconds,features_in,features_kept,features_dropped,columns_out,columns_per_feature,buckets_total,buckets_mean,buckets_median,tau_c_mean,tau_c_median,tau_c_best_per_feature_mean,tau_c_top100_sum
arm,,,,,,,,,,,,,,,
A1 one-vs-rest,OneVsRestCarver,0.05,481.5,435,435,0,823,1.892,1707,2.074,2.0,0.000488,0.000346,0.000717,0.294796
A2 multiclass,MulticlassCarver,0.05,139.2,435,420,15,420,1.000,865,2.060,2.0,0.000266,0.000133,0.000266,0.254045
A3 ordinal,OrdinalCarver,0.05,126.8,435,397,38,397,1.000,834,2.101,2.0,0.000590,0.000459,0.000590,0.262984
"A4 ordinal, Wilson off",OrdinalCarver,1.00,118.7,435,397,38,397,1.000,833,2.098,2.0,0.000590,0.000447,0.000590,0.263350


### §3.2 — target geometry

In [9]:
# §3.2 — target geometry: what each arm bought, and what it cost in columns
geometry = summary_table.loc[
    ["A1 one-vs-rest", "A2 multiclass", "A3 ordinal"],
    [
        "carve_seconds",
        "columns_out",
        "columns_per_feature",
        "buckets_total",
        "tau_c_mean",
        "tau_c_best_per_feature_mean",
        "tau_c_top100_sum",
    ],
]
print(geometry.to_string())

reference = summary_table.loc["A3 ordinal"]
for arm in ("A1 one-vs-rest", "A2 multiclass"):
    row = summary_table.loc[arm]
    print(
        f"\n{arm} vs A3 ordinal: "
        f"{row['columns_out'] / reference['columns_out']:.2f}x the columns, "
        f"best-per-feature tau_c {row['tau_c_best_per_feature_mean']:.6f} "
        f"vs {reference['tau_c_best_per_feature_mean']:.6f}"
    )

                carve_seconds  columns_out  columns_per_feature  buckets_total  tau_c_mean  tau_c_best_per_feature_mean  tau_c_top100_sum
arm                                                                                                                                      
A1 one-vs-rest          481.5          823                1.892           1707    0.000488                     0.000717          0.294796
A2 multiclass           139.2          420                1.000            865    0.000266                     0.000266          0.254045
A3 ordinal              126.8          397                1.000            834    0.000590                     0.000590          0.262984

A1 one-vs-rest vs A3 ordinal: 2.07x the columns, best-per-feature tau_c 0.000717 vs 0.000590

A2 multiclass vs A3 ordinal: 1.06x the columns, best-per-feature tau_c 0.000266 vs 0.000590


### §3.3 — bucket-frequency policy

In [10]:
# §3.3 — Wilson-score CI on (A3) vs the hard cutoff (A4)
wilson = summary_table.loc[
    ["A3 ordinal", "A4 ordinal, Wilson off"],
    [
        "carve_seconds",
        "features_dropped",
        "columns_out",
        "buckets_total",
        "buckets_mean",
        "tau_c_mean",
        "tau_c_best_per_feature_mean",
    ],
]
print(wilson.to_string())

on = detail_table[detail_table["arm"] == "A3 ordinal"].set_index("column")
off = detail_table[detail_table["arm"] == "A4 ordinal, Wilson off"].set_index("column")
shared = on.index.intersection(off.index)

delta = off.loc[shared, "n_buckets"] - on.loc[shared, "n_buckets"]
print(f"\nfeatures carved by both arms: {len(shared)}")
print(f"  same bucket count: {(delta == 0).sum()}")
print(f"  more buckets without the Wilson test: {(delta > 0).sum()}")
print(f"  fewer buckets without it: {(delta < 0).sum()}")
print(f"  net extra buckets admitted by the hard cutoff: {int(delta.sum())}")

tau_delta = off.loc[shared, "tau_c"] - on.loc[shared, "tau_c"]
print(
    f"\ndev tau_c, hard cutoff minus Wilson: mean {tau_delta.mean():+.6f}, "
    f"median {tau_delta.median():+.6f}, better on {(tau_delta > 0).sum()} of {len(shared)} features"
)

                        carve_seconds  features_dropped  columns_out  buckets_total  buckets_mean  tau_c_mean  tau_c_best_per_feature_mean
arm                                                                                                                                       
A3 ordinal                      126.8                38          397            834         2.101     0.00059                      0.00059
A4 ordinal, Wilson off          118.7                38          397            833         2.098     0.00059                      0.00059

features carved by both arms: 397
  same bucket count: 394
  more buckets without the Wilson test: 1
  fewer buckets without it: 2
  net extra buckets admitted by the hard cutoff: -1

dev tau_c, hard cutoff minus Wilson: mean -0.000000, median +0.000000, better on 4 of 397 features


## Reading these numbers

`tau_c` is measured on **dev**, so a bucketing that only works on train scores low here —
that is the point of measuring it out of sample rather than reporting each carver's own
training association.

What this pass does **not** answer: whether a higher tau-c survives feature selection and
XGBoost into a better dev log loss. Column count interacts with the selection budget
(§3.5), so a geometry that scores well per column can still lose once a fixed budget has to
be spread across many more of them. That needs the downstream arms — same selector, same
seeded Optuna budget — not this notebook.